# 06 · DistilBERT (rama de texto principal) — Grupo 17 · Sprint 3 · v1.1
**GPU obligatoria** (Colab: Entorno → T4 · Kaggle: Accelerator → GPU T4/P100, Internet ON).

Flujo automático, **reanudable**: cada fold guarda `results/distilbert_v1_<fold>.json` al terminar y, si el notebook
se corta (cuota de GPU), al relanzarlo salta lo ya hecho.
1. **Humo**: 30 pasos con 512 correos. Si falla, se detiene aquí con el error (no gasta GPU en vano).
2. **Principal**: train → val (umbral de Recall 95 %) → test. Guarda modelo, probabilidades de val/test (para la fusión) y latencia en CPU.
3. **Folds** fuera de distribución: phishing_pot, Nigerian_Fraud, Nazario anual y temporal 2025+.

Criterio del Sprint 3 (D8 de la bitácora): mejorar el recall fuera de distribución del baseline
(phishing_pot 26,5 % · Nigerian 26,8 % · Nazario 84,3 % · temporal 65,9 %).

In [1]:
# Celda 1: Entorno, rutas y configuración
import os, sys, json, time, random, glob
from pathlib import Path
import numpy as np, pandas as pd
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive")
    BASE_IN = BASE_OUT = Path("/content/drive/MyDrive/PI_G17")
elif ENV == "kaggle":   # dataset privado con emails_v1.parquet (y opcionalmente results/baseline_v1.json)
    hallado = sorted(Path("/kaggle/input").rglob("emails_v1.parquet"))
    assert hallado, "Adjunta el dataset privado que contiene emails_v1.parquet (Add Input)"
    BASE_IN, BASE_OUT = hallado[0].parent, Path("/kaggle/working/PI_G17")
else:
    BASE_IN = BASE_OUT = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
PARQUET = BASE_IN / "emails_v1.parquet" if ENV == "kaggle" else BASE_IN / "data/processed/emails_v1.parquet"
RES, MODELS, PROC = BASE_OUT / "results", BASE_OUT / "models", BASE_OUT / "data/processed"
for p in (RES, MODELS, PROC): p.mkdir(parents=True, exist_ok=True)

CFG = {"modelo": "distilbert-base-uncased", "max_len": 256, "batch": 32, "epocas": 3, "lr": 5e-5,
       "warmup": 0.1, "semilla": 17, "paciencia": 1,
       # En Colab se puede correr todo; si la cuota se acaba, relanzar continúa donde quedó.
       "folds": ["principal", "loso_phishing_pot", "loso_Nigerian_Fraud", "loso_nazario_anual", "temporal_phishing_2025+"],
       "humo": {"n": 512, "pasos": 30}}
MARCADORES = ["__url__", "__ip__", "__email__", "__anio__", "__dominio_propio__"]   # = data_prep.MARCADORES
CLASES = ["legitimo", "spam", "phishing"]; PH = 2
print(ENV, "| entrada:", PARQUET, "| salida:", BASE_OUT)

Mounted at /content/drive
colab | entrada: /content/drive/MyDrive/PI_G17/data/processed/emails_v1.parquet | salida: /content/drive/MyDrive/PI_G17


In [2]:
# Celda 2: Dependencias, semillas y GPU
%pip install -q "transformers>=4.40" accelerate
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
import transformers
def fijar_semillas(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s); os.environ["PYTHONHASHSEED"] = str(s)
fijar_semillas(CFG["semilla"])
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEV.type == "cuda" or ENV == "local", "Sin GPU: cambia el tipo de entorno a GPU antes de seguir"
GPU = torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "|", GPU)
D = pd.read_parquet(PARQUET)
D = D.reset_index(drop=True)
print("correos:", len(D))

torch 2.11.0+cu130 | transformers 5.17.0 | Tesla T4
correos: 54684


In [3]:
# Celda 3: Métricas (idénticas al baseline 05, para comparar)
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix
def umbral_para_recall(y, p_ph, objetivo=0.95):
    s = np.sort(p_ph[y == PH])[::-1]
    return float(s[min(int(np.ceil(objetivo * len(s))) - 1, len(s) - 1)]) if len(s) else 0.5
def predecir(P, umbral):
    pred = P[:, :2].argmax(1); pred[P[:, PH] >= umbral] = PH; return pred
def metricas(y, pred):
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    no_ph = y != PH
    return {"f1_macro": round(f1_score(y, pred, average="macro", labels=[0, 1, 2], zero_division=0), 4),
            "precision": dict(zip(CLASES, np.round(p, 4).tolist())), "recall": dict(zip(CLASES, np.round(r, 4).tolist())),
            "fpr_phishing": round(float((pred[no_ph] == PH).mean()), 4) if no_ph.any() else None,
            "fpr_legitimo_a_phishing": round(float((pred[y == 0] == PH).mean()), 4) if (y == 0).any() else None,
            "n": int(len(y))}
def particion(fold):
    """Máscaras train/val/test y 'fuera' según las columnas creadas en 03_preparacion_splits."""
    s = D["split"] if fold == "principal" else D[f"fold_{fold}"]
    fuera = np.zeros(len(D), bool) if fold == "principal" else D[f"fuera_{fold}"].values
    return (s == "train").values, (s == "val").values, (s == "test").values, fuera

In [4]:
# Celda 4: Entrenamiento y predicción
TOK = AutoTokenizer.from_pretrained(CFG["modelo"])
TOK.add_tokens(MARCADORES)                      # los marcadores de data_prep pasan a ser tokens únicos
def codificar(idx):
    enc = TOK(D.loc[idx, "texto_modelo"].tolist(), truncation=True, max_length=CFG["max_len"])
    return [{"input_ids": a, "attention_mask": m, "labels": int(y)}
            for a, m, y in zip(enc["input_ids"], enc["attention_mask"], D.loc[idx, "y"])]
COL = DataCollatorWithPadding(TOK)
def nuevo_modelo():
    m = AutoModelForSequenceClassification.from_pretrained(CFG["modelo"], num_labels=3)
    m.resize_token_embeddings(len(TOK)); return m.to(DEV)
@torch.no_grad()
def probabilidades(m, idx, bs=128):
    m.eval(); out = []
    for b in DataLoader(codificar(idx), batch_size=bs, collate_fn=COL):
        b = {k: v.to(DEV) for k, v in b.items() if k != "labels"}
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
            out.append(torch.softmax(m(**b).logits.float(), -1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 3))
def entrenar(idx_tr, idx_va, max_pasos=None, epocas=None):
    fijar_semillas(CFG["semilla"]); m = nuevo_modelo()
    y_tr = D.loc[idx_tr, "y"].values
    pesos = torch.tensor(len(y_tr) / (3 * np.bincount(y_tr, minlength=3)), dtype=torch.float, device=DEV)  # class weights
    dl = DataLoader(codificar(idx_tr), batch_size=CFG["batch"], shuffle=True, collate_fn=COL,
                    generator=torch.Generator().manual_seed(CFG["semilla"]))
    ep = epocas or CFG["epocas"]; total = max_pasos or len(dl) * ep
    opt = torch.optim.AdamW(m.parameters(), lr=CFG["lr"], weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * total), total)
    scaler = torch.cuda.amp.GradScaler(enabled=DEV.type == "cuda")
    perdida = torch.nn.CrossEntropyLoss(weight=pesos)
    mejor, mejor_estado, sin_mejora, paso, hist = -1, None, 0, 0, []
    for e in range(ep):
        m.train(); t0 = time.time()
        for b in dl:
            b = {k: v.to(DEV) for k, v in b.items()}; y = b.pop("labels")
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
                loss = perdida(m(**b).logits.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); paso += 1
            if max_pasos and paso >= max_pasos: break
        if max_pasos:                                   # humo: sin validación
            return m, [{"pasos": paso, "loss_final": round(float(loss), 4)}]
        Pv = probabilidades(m, idx_va); f1v = f1_score(D.loc[idx_va, "y"], Pv.argmax(1), average="macro")
        hist.append({"epoca": e + 1, "f1_val": round(float(f1v), 4), "loss": round(float(loss), 4), "s": round(time.time() - t0)})
        print(hist[-1])
        if f1v > mejor:
            mejor, sin_mejora = f1v, 0
            mejor_estado = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        else:
            sin_mejora += 1
            if sin_mejora > CFG["paciencia"]: break
    m.load_state_dict(mejor_estado); return m, hist

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
# Celda 5: Humo (se detiene aquí si algo falla)
t0 = time.time()
tr, va, te, _ = particion("principal")
rng = np.random.RandomState(CFG["semilla"])
idx_humo = rng.choice(np.where(tr)[0], CFG["humo"]["n"], replace=False)
m_humo, h = entrenar(idx_humo, None, max_pasos=CFG["humo"]["pasos"])
P_humo = probabilidades(m_humo, np.where(va)[0][:256])
assert P_humo.shape == (256, 3) and np.allclose(P_humo.sum(1), 1, atol=1e-3), "Probabilidades inválidas"
HUMO = {"ok": True, "pasos": h[0]["pasos"], "loss_final": h[0]["loss_final"], "segundos": round(time.time() - t0), "gpu": GPU}
del m_humo; torch.cuda.empty_cache() if DEV.type == "cuda" else None
print("HUMO OK ->", HUMO)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion

HUMO OK -> {'ok': True, 'pasos': 16, 'loss_final': 0.6816, 'segundos': 22, 'gpu': 'Tesla T4'}


In [6]:
# Celda 6: Principal + folds (reanudable)
def correr(fold):
    destino = RES / f"distilbert_v1_{fold}.json"
    probs = PROC / ("text_scores_distilbert_v1.parquet" if fold == "principal" else f"text_scores_distilbert_v1_{fold}.parquet")
    if destino.exists() and probs.exists():        # v1.1: un fold solo está hecho si también guardó sus probabilidades
        print(f"{fold}: ya existe, se omite"); return json.loads(destino.read_text())
    t0 = time.time(); tr, va, te, fuera = particion(fold)
    itr, iva, ite = np.where(tr)[0], np.where(va)[0], np.where(te)[0]
    m, hist = entrenar(itr, iva)
    Pv, Pt = probabilidades(m, iva), probabilidades(m, ite)
    u = umbral_para_recall(D.loc[iva, "y"].values, Pv[:, PH]); yt = D.loc[ite, "y"].values
    R = {"fold": fold, "historial": hist, "umbral_val_recall95": round(u, 4),
         "test_argmax": metricas(yt, Pt.argmax(1)), "test_umbral_recall95": metricas(yt, predecir(Pt, u)),
         "segundos": round(time.time() - t0), "gpu": GPU, "cfg": {k: v for k, v in CFG.items() if k != "folds"}}
    if fold == "principal":
        R["matriz_confusion_test_umbral"] = confusion_matrix(yt, predecir(Pt, u), labels=[0, 1, 2]).tolist()
        sin_nig = (D.loc[ite, "grupo"] != "Nigerian_Fraud").values
        R["test_umbral_sin_nigerian"] = metricas(yt[sin_nig], predecir(Pt[sin_nig], u))
        R["test_recall_phishing_por_grupo"] = {g: round(float((predecir(Pt, u)[(D.loc[ite, "grupo"] == g).values] == PH).mean()), 4)
                                               for g in sorted(D.loc[ite][D.loc[ite, "clase"] == "phishing"]["grupo"].unique())}
        destino_m = MODELS / "distilbert_v1"; m.save_pretrained(destino_m); TOK.save_pretrained(destino_m)
        pd.DataFrame({"id": D.loc[np.r_[iva, ite], "id"].values, "split": ["val"] * len(iva) + ["test"] * len(ite),
                      "p_legitimo": np.r_[Pv[:, 0], Pt[:, 0]], "p_spam": np.r_[Pv[:, 1], Pt[:, 1]],
                      "p_phishing": np.r_[Pv[:, 2], Pt[:, 2]]}).to_parquet(PROC / "text_scores_distilbert_v1.parquet", index=False)
        # Latencia de inferencia en CPU, un correo a la vez (KPI < 1 s)
        mc = m.to("cpu").eval(); muestra = D.loc[ite[:100], "texto_modelo"].tolist(); t1 = time.perf_counter()
        with torch.no_grad():
            for t in muestra: mc(**TOK(t, truncation=True, max_length=CFG["max_len"], return_tensors="pt"))
        R["latencia_cpu_ms_por_correo"] = round(1000 * (time.perf_counter() - t1) / len(muestra), 1)
    else:
        sf = fuera[te]
        R["solo_lo_fuera"] = metricas(yt[sf], predecir(Pt, u)[sf])
        # v1.1: probabilidades del fold (val y test) para evaluar la FUSIÓN fuera de distribución en 07
        pd.DataFrame({"id": D.loc[np.r_[iva, ite], "id"].values, "split": ["val"] * len(iva) + ["test"] * len(ite),
                      "p_legitimo": np.r_[Pv[:, 0], Pt[:, 0]], "p_spam": np.r_[Pv[:, 1], Pt[:, 1]],
                      "p_phishing": np.r_[Pv[:, 2], Pt[:, 2]]}).to_parquet(probs, index=False)
    destino.write_text(json.dumps(R, indent=1, ensure_ascii=False, default=str))
    del m; torch.cuda.empty_cache() if DEV.type == "cuda" else None
    print(fold, "->", R.get("solo_lo_fuera", R["test_umbral_recall95"])["recall"], f'{R["segundos"]}s')
    return R
RESULTADOS = {f: correr(f) for f in CFG["folds"]}

principal: ya existe, se omite


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_3451/2195803916.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEV.type 

{'epoca': 1, 'f1_val': 0.9837, 'loss': 0.0012, 's': 231}
{'epoca': 2, 'f1_val': 0.9927, 'loss': 0.0018, 's': 233}
{'epoca': 3, 'f1_val': 0.9924, 'loss': 0.0001, 's': 231}
loso_phishing_pot -> {'legitimo': 0.0, 'spam': 0.0, 'phishing': 0.4425} 825s


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_3451/2195803916.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEV.type 

{'epoca': 1, 'f1_val': 0.983, 'loss': 0.0032, 's': 243}
{'epoca': 2, 'f1_val': 0.9883, 'loss': 0.0012, 's': 241}
{'epoca': 3, 'f1_val': 0.9884, 'loss': 0.0002, 's': 241}
loso_nazario_anual -> {'legitimo': 0.0, 'spam': 0.0, 'phishing': 0.7824} 830s


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_3451/2195803916.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEV.type 

{'epoca': 1, 'f1_val': 0.9831, 'loss': 0.381, 's': 242}
{'epoca': 2, 'f1_val': 0.9863, 'loss': 0.0019, 's': 240}
{'epoca': 3, 'f1_val': 0.9892, 'loss': 0.0004, 's': 240}
temporal_phishing_2025+ -> {'legitimo': 0.0, 'spam': 0.0, 'phishing': 0.8289} 826s


In [7]:
# Celda 7: Comparación con el baseline y RESUMEN
base = None
for cand in (BASE_OUT / "results/baseline_v1.json", BASE_IN / "baseline_v1.json"):
    if Path(cand).exists(): base = json.loads(Path(cand).read_text()); break
comp = {}
for f, R in RESULTADOS.items():
    if f == "principal":
        comp[f] = {"distilbert_f1": R["test_umbral_recall95"]["f1_macro"], "distilbert_recall_ph": R["test_umbral_recall95"]["recall"]["phishing"],
                   "baseline_f1": base["test_umbral_recall95"]["f1_macro"] if base else None,
                   "baseline_recall_ph": base["test_umbral_recall95"]["recall"]["phishing"] if base else None}
    else:
        comp[f] = {"distilbert_recall_fuera": R["solo_lo_fuera"]["recall"]["phishing"],
                   "baseline_recall_fuera": base["folds"][f]["solo_lo_fuera"]["recall"]["phishing"] if base and f in base.get("folds", {}) else None}
RESUMEN = {"entorno": ENV, "gpu": GPU, "humo": HUMO, "comparacion": comp,
           "principal": {k: RESULTADOS["principal"].get(k) for k in ("historial", "umbral_val_recall95", "test_argmax", "test_umbral_recall95",
                                                                     "test_umbral_sin_nigerian", "test_recall_phishing_por_grupo",
                                                                     "matriz_confusion_test_umbral", "latencia_cpu_ms_por_correo", "segundos")}}
(RES / "distilbert_v1_resumen.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (06) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (06) =====
{"entorno":"colab","gpu":"Tesla T4","humo":{"ok":true,"pasos":16,"loss_final":0.6816,"segundos":22,"gpu":"Tesla T4"},"comparacion":{"principal":{"distilbert_f1":0.983,"distilbert_recall_ph":0.9483,"baseline_f1":0.9846,"baseline_recall_ph":0.9617},"loso_phishing_pot":{"distilbert_recall_fuera":0.4425,"baseline_recall_fuera":0.2653},"loso_Nigerian_Fraud":{"distilbert_recall_fuera":0.1207,"baseline_recall_fuera":0.2681},"loso_nazario_anual":{"distilbert_recall_fuera":0.7824,"baseline_recall_fuera":0.8432},"temporal_phishing_2025+":{"distilbert_recall_fuera":0.8289,"baseline_recall_fuera":0.6588}},"principal":{"historial":[{"epoca":1,"f1_val":0.9841,"loss":0.0034,"s":255},{"epoca":2,"f1_val":0.988,"loss":0.0009,"s":268},{"epoca":3,"f1_val":0.9893,"loss":0.0003,"s":265}],"umbral_val_recall95":0.9946,"test_argmax":{"f1_macro":0.9899,"precision":{"legitimo":0.9932,"spam":0.9948,"phishing":0.9793},"recall":{"legitimo":0.9959,"spam":0.9889,"phishing":0.9872},"fpr_phishi